<a href="https://colab.research.google.com/github/Niamh3521/SatelliteDataAI-UOA/blob/main/Question6onlycode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
!pip install earthengine-api geemap mapclassify
!pip install -U geemap
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, cohen_kappa_score

# Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='geog761-project')

In [42]:
# Code to get you started
import zipfile
import geopandas as gpd

# Upload the ZIP manually using the Colab UI
from google.colab import files
uploaded = files.upload()  # <- Expects a ZIP

# Unzip
with zipfile.ZipFile("LCDB_v5.zip", 'r') as zip_ref: #<- Check file names
    zip_ref.extractall("lcdb")

# Read shapefile
gdf = gpd.read_file("lcdb/lcdb-v50-land-cover-database-version-50-mainland-new-zealand.shp") #<- Check file names
print(gdf.head())


Saving LCDB_v5.zip to LCDB_v5 (1).zip
           Name_2018          Name_2012          Name_2008          Name_2001  \
0  Indigenous Forest  Indigenous Forest  Indigenous Forest  Indigenous Forest   
1     Sand or Gravel     Sand or Gravel     Sand or Gravel     Sand or Gravel   
2  Indigenous Forest  Indigenous Forest  Indigenous Forest  Indigenous Forest   
3     Sand or Gravel     Sand or Gravel     Sand or Gravel     Sand or Gravel   
4  Indigenous Forest  Indigenous Forest  Indigenous Forest  Indigenous Forest   

           Name_1996  Class_2018  Class_2012  Class_2008  Class_2001  \
0  Indigenous Forest          69          69          69          69   
1     Sand or Gravel          10          10          10          10   
2  Indigenous Forest          69          69          69          69   
3     Sand or Gravel          10          10          10          10   
4  Indigenous Forest          69          69          69          69   

   Class_1996  ... Wetland_96 Onshore_18 O

In [43]:
# Define area of interest
aoi = ee.Geometry.Rectangle([175.30643, -36.34921, 175.55101, -36.04988])
Map = geemap.Map(center=[-36.3055, 175.5433], zoom=10)
Map.addLayer(aoi, {}, 'AOI')

Map

Map(center=[-36.3055, 175.5433], controls=(WidgetControl(options=['position', 'transparent_bg'], position='top…

In [44]:
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
           qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask).divide(10000).select(['B2', 'B3', 'B4', 'B8'])

# Now build the clean collection with selected bands
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate('2018-12-01', '2019-02-28')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
      .map(mask_s2_clouds)
      .median()
      .clip(aoi))

In [45]:
import json

# 1. Convert GeoDataFrame (WGS84) to Earth Engine FeatureCollection
gdf_wgs84 = gdf.to_crs(epsg=4326)
geojson_str = gdf_wgs84[['Class_2018', 'Name_2018', 'geometry']].to_json()
ee_fc = ee.FeatureCollection(json.loads(geojson_str))

# Define AOI from the vector bounds
aoi = ee_fc.geometry().bounds()

# 2. Rasterize LCDB polygons directly to an ee.Image — original class codes, no remapping
landcover = ee_fc.reduceToImage(
    properties=['Class_2018'],
    reducer=ee.Reducer.first()
).rename('landcover').clip(aoi)

In [46]:
training_data = s2.addBands(landcover).addBands(landcover)

In [47]:
bands = ['B2', 'B3', 'B4', 'B8']
sample = training_data.select(bands + ['landcover']).sample(
    region=aoi,
    scale=10,
    numPixels=100000,
    seed=2,
    geometries=True)

class_counts = sample.reduceColumns(
    reducer=ee.Reducer.frequencyHistogram(),
    selectors=['landcover'])

print(class_counts.getInfo())

{'histogram': {'1': 78, '10': 733, '12': 1, '16': 97, '2': 22, '21': 9, '22': 118, '33': 16, '40': 1849, '41': 296, '45': 346, '46': 160, '5': 2, '51': 75, '52': 17213, '54': 5006, '6': 4, '64': 7, '69': 7325, '70': 147, '71': 157}}


In [48]:
# Add random column
sample = sample.randomColumn('random')

# Split
train = sample.filter(ee.Filter.lt('random', 0.7)) # 70% of the data to train model
valid = sample.filter(ee.Filter.And(ee.Filter.gte('random', 0.7), ee.Filter.lt('random', 0.9))) #20% of the data to valadate model
test = sample.filter(ee.Filter.gte('random', 0.9)) # 10% OF THE DATA TO TEST DATA only happends once and cant be reused

In [49]:
def fc_to_lists(fc, classProp, predProp):
    values = fc.aggregate_array(classProp).getInfo()
    preds = fc.aggregate_array(predProp).getInfo()
    return values, preds

In [50]:
# mask the land
land_mask = landcover.reproject(crs='EPSG:4326', scale=10).mask()


# labels

raw_classes_py = sorted(gdf['Class_2018'].unique().tolist())
id_to_name = gdf.drop_duplicates('Class_2018').set_index('Class_2018')['Name_2018'].to_dict()
label_names = [id_to_name[c] for c in raw_classes_py]


# SVM — train, classify, evaluate
class_property = 'landcover'
svm = ee.Classifier.libsvm(kernelType='RBF', gamma=0.5, cost=10).train(
    features=train,
    classProperty=class_property,
    inputProperties=bands
)

# Classify test set
test_classified = test.classify(svm, 'predicted')

# Pull predictions down to Python
y_true_svm, y_pred_svm = fc_to_lists(test_classified, class_property, 'predicted')

# Classify full image
svm_classified_map = (
    s2.select(bands)
    .classify(svm, 'classification')
    .updateMask(land_mask)
)


# METRICS — using actual LCDB codes as labels, not 1-N

cm_svm = confusion_matrix(y_true_svm, y_pred_svm, labels=raw_classes_py)
report_svm = classification_report(
    y_true_svm, y_pred_svm,
    labels=raw_classes_py,
    target_names=[str(l) for l in label_names],
    zero_division=0
)

print("=== SVM — Test Set ===")
print(pd.DataFrame(cm_svm, index=[f"Actual {l}" for l in label_names],
                           columns=[f"Pred {l}" for l in label_names]))
print("\n", report_svm)
print(f"Accuracy: {accuracy_score(y_true_svm, y_pred_svm):.3f}")
print(f"Kappa: {cohen_kappa_score(y_true_svm, y_pred_svm):.3f}")

=== SVM — Test Set ===
                                                  Pred Built-up Area (settlement)  \
Actual Built-up Area (settlement)                                               0   
Actual Urban Parkland/Open Space                                                0   
Actual Transport Infrastructure                                                 0   
Actual Surface Mine or Dump                                                     0   
Actual Sand or Gravel                                                           0   
Actual Landslide                                                                0   
Actual Gravel or Rock                                                           0   
Actual Lake or Pond                                                             0   
Actual River                                                                    0   
Actual Estuarine Open Water                                                     0   
Actual Orchard, Vineyard or Other Perennia

In [51]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors

num_classes = len(raw_classes_py)

if num_classes <= 10:
    cmap = cm.get_cmap('tab10', num_classes)
elif num_classes <= 20:
    cmap = cm.get_cmap('tab20', num_classes)
else:
    cmap = cm.get_cmap('nipy_spectral', num_classes)

palette = [mcolors.to_hex(cmap(i)) for i in range(num_classes)]

vis_params = {
    'min': min(raw_classes_py),
    'max': max(raw_classes_py),
    'palette': palette
}


In [52]:
sample_check = svm_classified_map.sample(
    region=aoi,
    scale=10,
    numPixels=2000,   # a sample, not every pixel
    seed=1
)
present_class_ids = sorted(set(sample_check.aggregate_array('classification').getInfo()))
print(present_class_ids)

[10, 40, 52, 69]


In [59]:
from ipyleaflet import WidgetControl
from ipywidgets import HTML



filtered_label_names = [id_to_name[c] for c in present_class_ids]
filtered_palette = [palette[raw_classes_py.index(c)] for c in present_class_ids]
legend_dict = {filtered_label_names[i]: filtered_palette[i] for i in range(len(present_class_ids))}

print(legend_dict)  # sanity check the actual class names

Map = geemap.Map()
Map.centerObject(aoi, 11)
Map.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'S2 RGB Composite')
Map.addLayer(svm_classified_map, vis_params, 'SVM Landcover Classification')
Map.addLayer(landcover, vis_params, 'LCDB Reference')
Map.add_legend(title="Landcover Classes (SVM)", legend_dict=legend_dict)
north_arrow_html = HTML(
    value='<div style="font-size:32px; text-align:center;">⬆<br><b style="font-size:14px;">N</b></div>'
)
north_arrow_control = WidgetControl(widget=north_arrow_html, position='topright')
Map.add_control(north_arrow_control)

Map


{'Sand or Gravel': '#0000d3', 'High Producing Exotic Grassland': '#00cb00', 'Manuka and/or Kanuka': '#ffc000', 'Indigenous Forest': '#db0000'}


Map(center=[-36.187073889065566, 175.41640470768527], controls=(WidgetControl(options=['position', 'transparen…